# 08 — Hypothesis Testing Theory
**References:** Casella & Berger (2002) Ch. 8 · Lehmann & Romano (2005) Ch. 3–4

## Narrative thread
```
Hypotheses -> Error types -> Power function -> NP lemma -> UMP tests -> p-values
```

## 1. Setup and error types

A **hypothesis test** is a decision rule based on data: reject $H_0$ (null) in favor of $H_1$ (alternative).

A test is characterized by its **critical region** $\mathcal{C}$: reject $H_0$ if $\mathbf{X} \in \mathcal{C}$.

| Decision | $H_0$ true | $H_1$ true |
|---|---|---|
| Reject $H_0$ | **Type I error** (prob = $\alpha$) | Correct (prob = Power) |
| Fail to reject $H_0$ | Correct (prob = $1-\alpha$) | **Type II error** (prob = $\beta$) |

**Power function:**
$$\beta(\theta) = P_\theta(\mathbf{X} \in \mathcal{C}) = \begin{cases} \alpha & \theta \in \Theta_0 \\ 1 - \text{Type II error} & \theta \in \Theta_1 \end{cases}$$

**Size:** $\alpha = \sup_{\theta \in \Theta_0} \beta(\theta)$ — the largest Type I error rate over all null parameter values.

**Ideal test:** $\beta(\theta) = \alpha$ for $\theta \in \Theta_0$ and $\beta(\theta) = 1$ for $\theta \in \Theta_1$. This is generally unattainable — we seek the best achievable test.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── Power function for z-test: H0: mu=0 vs H1: mu != 0 ───────────────────
sigma = 1.0
ns    = [10, 30, 100]
alpha = 0.05
mus   = np.linspace(-3, 3, 300)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for n in ns:
    se = sigma / np.sqrt(n)
    z_crit = stats.norm.ppf(1 - alpha/2)
    power = 1 - stats.norm.cdf(z_crit - mus/se) + stats.norm.cdf(-z_crit - mus/se)
    ax.plot(mus, power, lw=2, label=f'n={n}')

ax.axhline(alpha, color='gray', lw=1.5, linestyle='--', label=f'Size α={alpha}')
ax.axhline(0.80,  color='#f72585', lw=1,   linestyle=':',  label='80% power')
ax.axvline(0, color='k', lw=1, alpha=0.5)
ax.set_xlabel('$\\mu$'); ax.set_ylabel('Power $\\beta(\\mu)$')
ax.set_title('Power function: two-sided z-test\n$H_0: \\mu=0$ vs $H_1: \\mu\\neq 0$')
ax.legend()

# ── Type I/II tradeoff: shifting the critical value ───────────────────────
n_demo = 30
mu1    = 0.5  # alternative
alphas = np.linspace(0.001, 0.5, 300)
z_crits = stats.norm.ppf(1 - alphas)
powers  = 1 - stats.norm.cdf(z_crits - mu1 * np.sqrt(n_demo) / sigma)
type2   = 1 - powers

ax2 = axes[1]
ax2.plot(alphas, type2, color='#f72585', lw=2.5, label='Type II error (β)')
ax2.plot(alphas, alphas, color='#4361ee', lw=2, linestyle='--', label='Type I error (α) [reference]')
ax2.set_xlabel('Type I error (α)')
ax2.set_ylabel('Type II error (β)')
ax2.set_title(f'Error tradeoff: one-sided z-test, n={n_demo}, μ₁={mu1}\n'
              'Reducing α increases β — fundamental constraint')
ax2.legend()

plt.suptitle('Power Function and Error Types', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Neyman-Pearson Lemma

**Theorem (Neyman-Pearson, 1933):** For testing simple $H_0: \theta = \theta_0$ vs simple $H_1: \theta = \theta_1$, the **most powerful** (MP) test of size $\alpha$ has critical region:

$$\mathcal{C}^* = \left\{\mathbf{x} : \frac{f(\mathbf{x}\mid\theta_1)}{f(\mathbf{x}\mid\theta_0)} > k\right\}$$

where $k$ is chosen so that $P_{\theta_0}(\mathbf{X} \in \mathcal{C}^*) = \alpha$.

**The NP test is the likelihood ratio test** for simple hypotheses.

**Proof sketch:** For any other size-$\alpha$ test with power $\beta'$:
$$\beta^* - \beta' = E_{\theta_1}[\phi^* - \phi'] = \int (\phi^* - \phi)(f_1 - k f_0) \geq 0$$

The inequality follows because $\phi^* - \phi$ and $f_1 - kf_0$ have the same sign everywhere.

In [ ]:
# ── NP lemma: optimal region for Normal location shift ────────────────────
# H0: X ~ N(0,1),  H1: X ~ N(mu1, 1)
# LR = f1/f0 = exp(mu1*x - mu1^2/2) > k  <=>  x > c
# So the MP test is the one-sided z-test

mu0, mu1 = 0.0, 1.5
alpha     = 0.05
n_test    = 25
x_vals    = np.linspace(-4, 5, 400)

dist0 = stats.norm(mu0, 1/np.sqrt(n_test))
dist1 = stats.norm(mu1, 1/np.sqrt(n_test))
c_star = dist0.ppf(1 - alpha)  # NP critical value

power_np   = 1 - dist1.cdf(c_star)  # power of NP test (one-sided)
power_2s   = 1 - dist1.cdf(dist0.ppf(0.975)) + dist1.cdf(dist0.ppf(0.025))  # two-sided

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(x_vals, dist0.pdf(x_vals), color='#4361ee', lw=2.5, label=f'$H_0$: $\\bar X \\sim N({mu0}, 1/{n_test})$')
ax.plot(x_vals, dist1.pdf(x_vals), color='#f72585', lw=2.5, label=f'$H_1$: $\\bar X \\sim N({mu1}, 1/{n_test})$')
ax.fill_between(x_vals, dist0.pdf(x_vals), where=(x_vals > c_star),
                color='#4361ee', alpha=0.3, label=f'Type I: α={alpha}')
ax.fill_between(x_vals, dist1.pdf(x_vals), where=(x_vals > c_star),
                color='#f72585', alpha=0.3, label=f'Power={power_np:.3f}')
ax.axvline(c_star, color='k', lw=2, linestyle='--', label=f'Critical value c*={c_star:.2f}')
ax.set_xlabel('$\\bar{X}$')
ax.set_title(f'NP lemma: optimal one-sided test (n={n_test})\nH₀:μ=0 vs H₁:μ={mu1}')
ax.legend(fontsize=8)

ax2 = axes[1]
test_names   = ['NP one-sided\n(optimal)', 'Two-sided\n(sub-optimal here)']
test_powers  = [power_np, power_2s]
colors_bar   = ['#4361ee', '#f72585']
bars = ax2.bar(test_names, test_powers, color=colors_bar, alpha=0.8, width=0.4)
ax2.axhline(alpha, color='gray', lw=1.5, linestyle='--', label=f'Size α={alpha}')
for bar, pwr in zip(bars, test_powers):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{pwr:.3f}', ha='center', fontsize=11, fontweight='bold')
ax2.set_ylabel('Power')
ax2.set_ylim(0, 1)
ax2.set_title(f'Power comparison: NP (MP test) vs two-sided\n'
               f'NP gains {(power_np - power_2s)*100:.1f}% power at same α')
ax2.legend()

plt.suptitle('Neyman-Pearson Lemma — Most Powerful Test', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Uniformly Most Powerful (UMP) tests

A test is **UMP** of size $\alpha$ if it is the most powerful test of size $\alpha$ for every $\theta \in \Theta_1$.

**Theorem:** For one-sided hypotheses with a **monotone likelihood ratio** (MLR) family:
$$H_0: \theta \leq \theta_0 \text{ vs } H_1: \theta > \theta_0$$
the UMP test rejects for large values of the sufficient statistic $T(\mathbf{X})$.

**MLR families:** Exponential family distributions have MLR in the sufficient statistic.

**Limitation:** UMP tests generally do **not exist** for two-sided alternatives. For those, we use:
- Unbiased tests (UMPU)
- Likelihood ratio tests (notebook 09)

## 4. The p-value

The **p-value** is $p = P_{H_0}(T(\mathbf{X}) \geq T(\mathbf{x}_{\text{obs}}))$ — the probability of observing a test statistic at least as extreme as the one observed, under $H_0$.

**Key facts:**
- Under $H_0$: $p \sim \text{Uniform}(0,1)$ (for continuous distributions)
- $p < \alpha$ $\iff$ reject at level $\alpha$
- The p-value is **not** $P(H_0\text{ is true})$ — that requires Bayes' theorem

**Duality:** CIs and tests are dual — rejecting $H_0: \theta = \theta_0$ at level $\alpha$ $\iff$ $\theta_0$ is outside the $(1-\alpha)$ CI.

In [ ]:
# ── p-value distribution under H0 and H1 ──────────────────────────────────
n_reps  = 50_000
n_obs   = 30
mu0_pv  = 0.0
sigma   = 1.0

p_vals_null = []
p_vals_alt  = []

for _ in range(n_reps):
    # Under H0
    x = np.random.normal(mu0_pv, sigma, n_obs)
    t = (x.mean() - mu0_pv) / (sigma / np.sqrt(n_obs))
    p_vals_null.append(2 * (1 - stats.norm.cdf(abs(t))))

    # Under H1: mu = 0.5
    x = np.random.normal(0.5, sigma, n_obs)
    t = (x.mean() - mu0_pv) / (sigma / np.sqrt(n_obs))
    p_vals_alt.append(2 * (1 - stats.norm.cdf(abs(t))))

p_vals_null = np.array(p_vals_null)
p_vals_alt  = np.array(p_vals_alt)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(p_vals_null, bins=50, density=True, color='#4361ee', alpha=0.8, edgecolor='white')
axes[0].axhline(1, color='#f72585', lw=2, linestyle='--', label='Uniform(0,1)')
axes[0].set_title('p-values under $H_0$ — Uniform(0,1)\n'
                   'Rejecting at α=0.05 gives exactly 5% false positives')
axes[0].set_xlabel('p-value'); axes[0].legend()

axes[1].hist(p_vals_alt, bins=50, density=True, color='#f72585', alpha=0.8, edgecolor='white')
axes[1].axvline(0.05, color='k', lw=2, linestyle='--', label=f'α=0.05')
power_emp = (p_vals_alt < 0.05).mean()
axes[1].set_title(f'p-values under $H_1$ (μ=0.5, n=30)\n'
                   f'Fraction < 0.05 = power = {power_emp:.3f}')
axes[1].set_xlabel('p-value'); axes[1].legend()

plt.suptitle('p-value distribution: Uniform under H₀, skewed toward 0 under H₁',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Sample size and power analysis

For a two-sided z-test with effect size $\delta = \mu_1 - \mu_0$:
$$n = \frac{(z_{\alpha/2} + z_\beta)^2 \sigma^2}{\delta^2}$$

where $z_\beta = \Phi^{-1}(1-\beta)$ is the $z$-score for desired power $1-\beta$.

| Key takeaway | Statement |
|---|---|
| NP lemma | MP test = LR test for simple hypotheses |
| UMP tests | Exist for one-sided, MLR families |
| p-value | $\text{Uniform}(0,1)$ under $H_0$ — not $P(H_0)$ |
| CI-test duality | $p < \alpha$ $\iff$ $\theta_0 \notin$ $(1-\alpha)$ CI |

**Next:** notebook 09 — likelihood ratio tests, Wald, Score tests and their asymptotic chi-squared distribution.